# Train on a free GPU (Kaggle or Colab)

**Setup on Kaggle:** in the right-hand panel, set Accelerator to GPU T4 and turn Internet on.
Then run the cells in order.

If the session dies, run the setup cells again and use the resume cell. `last.pt` is saved every
epoch, but it only survives if you keep it, so copy it out of the session as you go, or re-upload
it to `checkpoints/` before resuming.

In [ ]:
REPO_URL = "https://github.com/nail-nail/manga-bubble-detector.git"

# GIT_TERMINAL_PROMPT=0 makes a wrong URL or private repo fail at once instead of hanging on a login prompt.
!GIT_TERMINAL_PROMPT=0 git clone --depth 1 {REPO_URL} repo
%cd repo
!pip install -q opencv-python-headless ImageHash huggingface_hub PyYAML tqdm

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU found. Turn on the GPU accelerator in the notebook settings."
print(torch.cuda.get_device_name(0))

## 1. Data

Downloads about 800 MB, then hashes every page and builds masks. Takes a few minutes.

In [ ]:
!python -m src.download_data
!python -m src.prepare_data

## 2. Sanity check

The loss must approach zero on one memorized batch. If it doesn't, stop and debug before spending GPU quota.

In [ ]:
!python -m src.train --overfit-batch --overfit-steps 200

## 3. Full training

Kaggle has 4 CPU cores, so 4 data-loading workers keep the GPU busy. Out of memory? Add `--set train.batch_size=4`. Running out of quota? Add `--set data.image_size=384`.

In [ ]:
!python -m src.train --set data.num_workers=4

### Resume after an interruption

Only run this if training was interrupted.

In [ ]:
# !python -m src.train --resume --set data.num_workers=4

## 4. Tune on validation

Picks `threshold`, `open_kernel`, `min_area` and `erode_px` on the validation set.

In [ ]:
!python -m src.evaluate --checkpoint checkpoints/best.pt --split val --tune
!python -m src.baseline --split val --tune

## 5. Final test evaluation

**Run this once, when you are done changing things.** If you go back and change the model after
seeing these numbers, the test set is no longer an honest estimate.

In [ ]:
# !python -m src.evaluate --checkpoint checkpoints/best.pt --split test --final
# !python -m src.baseline --split test --final

## 6. Download the results

Get `results.zip` from the output panel. It contains the checkpoint, logs and metrics.

In [ ]:
!zip -q -r /kaggle/working/results.zip checkpoints/best.pt checkpoints/best.postprocess.json reports -x 'reports/worst_*/*'
!ls -lh /kaggle/working/results.zip